# Laboratorium nr 7 - RAG z modelem lokalnym Ollama

In [8]:
import sys
import os
import json
import faiss
import pymupdf
import numpy as np
from tqdm import tqdm

from sentence_transformers import SentenceTransformer
import ollama

model_id = "gemma3:4b"

# Model embeddingowy – zamienia tekst na wektory liczbowe (embeddingi)
# all-mpnet-base-v2 produkuje wektory o wymiarze 768
embedder = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')

In [9]:
# Tworzymy pusty indeks FAISS typu FlatL2 – przechowuje wektory i szuka po odległości euklidesowej
# get_embedding_dimension() zwraca wymiar wektorów modelu embeddingowego (768)
index = faiss.IndexFlatL2(embedder.get_embedding_dimension())

# Lista słowników przechowująca metadane każdego chunka (nazwa pliku, numer strony, tekst)
metadata = []

print('Number of chunks: ', index.ntotal) # 0

Number of chunks:  0


In [10]:
class Utils:
    def __init__(self, embedding_model: SentenceTransformer=None, llm_model=None, index=None, metadata=None, chunk_size=512):
        self.embedding_model = embedding_model
        self.llm_model = llm_model
        self.index = index
        self.metadata = metadata
        # Rozmiar chunka w znakach – określa jak długie będą fragmenty tekstu
        self.chunk_size = chunk_size

    def extract_text_from_pdf(self, pdf_path):
        """
        Extract text from PDF file. Returns a list of tuples (page_number, text).
        """
        text = []
        pdf_document = pymupdf.open(pdf_path)
        for page_num in range(len(pdf_document)):
            page = pdf_document.load_page(page_num)
            # Zamieniamy znaki nowej linii na spacje, żeby tekst był ciągły
            text.append((page_num, str(page.get_text()).replace("\n", " ")))
        return text

    def chunk_text(self, text: list[tuple[int, str]]):
        chunks = []
        for page_num, page_text in text:
            # Dzielimy tekst strony na fragmenty o długości chunk_size znaków
            page_chunks = [
                (page_num, page_text[i:i+self.chunk_size])
                for i in range(0, len(page_text), self.chunk_size)
            ]
            chunks.extend(page_chunks)
        return chunks

    def add_chunks_to_faiss(self, chunks, filename, db_loc="vec_db/"):
        for chunk_num, (page_number, chunk) in enumerate(tqdm(chunks, desc="Adding chunks to FAISS")):
            # Zamieniamy tekst chunka na wektor embeddingowy
            embeddings = self.embedding_model.encode(chunk, show_progress_bar=False)
            # Dodajemy wektor do indeksu FAISS
            self.index.add(np.array([embeddings]))
            # Zapisujemy metadane chunka – powiążemy je z wektorem przez pozycję w indeksie
            self.metadata.append({
                "filename": filename,
                "page_number": page_number,
                "chunk_num": chunk_num,
                "chunk": chunk
            })
        # Zapisujemy indeks FAISS na dysk, żeby nie trzeba było go odbudowywać przy każdym uruchomieniu
        faiss.write_index(self.index, db_loc + "vector_database.index")
        with open(db_loc + "metadata.json", "w") as file:
            json.dump(self.metadata, file)

    def process_file(self, file_path):
        """
        Process the file and add chunks to FAISS index
        """
        if file_path.endswith('.pdf'):
            text = self.extract_text_from_pdf(file_path)
        else:
            print(f"Unsupported file format, with extension: {os.path.splitext(file_path)[1]}")
            return 0

        chunks = self.chunk_text(text)
        self.add_chunks_to_faiss(chunks, filename=os.path.basename(file_path))
        return len(chunks)
    
    def answer_question(self, prompt_template="", query="", max_tokens=512, temp=0.7, k=5):
        # Zamieniamy pytanie użytkownika na wektor embeddingowy
        question_embedding = self.embedding_model.encode(query, show_progress_bar=False)

        # Szukamy k najbliższych wektorów w FAISS – D to odległości, I to indeksy znalezionych chunków
        D, I = self.index.search(np.array([question_embedding]), k)
        # Pobieramy metadane (tekst) znalezionych chunków
        chunks = [self.metadata[i] for i in I[0]]

        # Sklejamy teksty chunków w jeden blok kontekstu dla modelu
        context = ""
        for i, chunk in enumerate(chunks):
            context += f"{i+1}. {chunk['chunk']}\n"

        # Wstawiamy kontekst i pytanie do szablonu promptu
        prompt = prompt_template.format(context=context, query=query)

        messages = [
            {
                "role": "system",
                "content": (
                    "Be helpful, straight to the point. "
                    "Use only context. Do not hallucinate."
                )
            },
            {"role": "user", "content": prompt},
        ]

        response = ollama.chat(
            model=self.llm_model,
            messages=messages,
            options={"temperature": temp, "num_predict": max_tokens}
        )
        answer = response['message']['content']

        return answer, chunks

In [11]:
# Tworzymy obiekt Utils łącząc wszystkie komponenty RAG w jednym miejscu
utils = Utils(
    embedder,   # model embeddingowy do zamiany tekstu na wektory
    model_id,   # LLM do generowania odpowiedzi
    index,      # indeks FAISS z wektorami chunków
    metadata,   # metadane chunków (tekst, strona, plik)
    chunk_size=512 # ilość treści w każdym fragmencie - zwiększamy by dać modelowi więcej treści gdy odmówi odpowiedzi
)

In [12]:
knowledge_dir = "knowledge/"
# Przetwarzamy każdy plik PDF z katalogu – dzielimy na chunki i dodajemy do FAISS
for file in os.listdir(knowledge_dir):
    utils.process_file(knowledge_dir + file)
    
print('Number of chunks: ', index.ntotal)

Adding chunks to FAISS: 100%|██████████| 428/428 [00:11<00:00, 38.25it/s]

Number of chunks:  913


In [13]:
# przukładowe pytania dotyczace powyzszych dokumentow
questions = [
    "Dlaczego astronomia kosmiczna jest ważna dla współczesnej nauki?",
    "Jak atmosfera Ziemi wpływa na obserwacje astronomiczne?",
    "Jakie rodzaje promieniowania elektromagnetycznego są wykorzystywane w astronomii?",
    "Czym różni się teleskop optyczny od radioteleskopu?",
    "Jakie informacje o obiektach kosmicznych można uzyskać dzięki analizie widma?",
    "Jak astronomowie wykorzystują podczerwień do badania kosmosu?",
    "Czym jest interferometria w astronomii?",
    "Jakie odkrycia umożliwiły obserwacje w zakresie fal radiowych?",
    "Czym są egzoplanety?",
    "Jakie są główne metody wykrywania egzoplanet?",
    "Dlaczego wykrywanie małych egzoplanet jest trudniejsze niż dużych?",
    "Czym jest strefa zamieszkiwalna wokół gwiazdy?",
    "Jak astronomowie badają atmosfery egzoplanet?",
    "Jakie cechy planety mogą wskazywać na możliwość istnienia życia?",
    "Jakie typy egzoplanet odkryto do tej pory?",
    "Jakie przyszłe technologie mogą poprawić wykrywanie egzoplanet?",
    "Czym jest astroML?",
    "Jak machine learning jest wykorzystywany w astronomii?",
    "Jakie typy danych astronomicznych analizuje się za pomocą ML?",
    "Dlaczego astronomia generuje duże ilości danych?",
    "Jakie algorytmy klasyfikacji są stosowane w analizie danych astronomicznych?",
    "Na czym polega klasteryzacja obiektów astronomicznych?",
    "W jaki sposób wykrywa się anomalie w danych astronomicznych?",
    "Jakie znaczenie mają sieci neuronowe w analizie obrazów kosmosu?",
    "Jak działa analiza szeregów czasowych w obserwacjach astronomicznych?",
    "Czym różni się supervised learning od unsupervised learning w kontekście astronomii?",
    "Jak ML pomaga w klasyfikacji galaktyk?",
    "Czym zajmuje się astrochemia?",
    "Jak powstają cząsteczki w przestrzeni międzygwiazdowej?",
    "Jaką rolę odgrywa pył kosmiczny w formowaniu gwiazd i planet?",
    "Dlaczego lód międzygwiazdowy jest ważny dla chemii kosmosu?",
    "Jakie związki organiczne odkryto w obłokach molekularnych?",
    "Jakie procesy zachodzą w dyskach protoplanetarnych?",
    "Jakie znaczenie ma astrochemia dla badań nad pochodzeniem życia?",
    "Jakie techniki spektroskopowe stosuje się w astrochemii?",
    "Jakie są główne źródła pyłu międzygwiazdowego?",
    "Jak astrochemia pomaga badać atmosfery egzoplanet?",
]

In [15]:
from IPython.display import display, Markdown

prompt_template ="""Based on the following context items, please answer the query.
Give yourself room to think by extracting relevant passages from the context before answering the query.
Don't return the thinking, only return the answer.
Answer in Polish language only.
Use the following examples as reference for the ideal answer style.
Example 1:
Pytanie: Dlaczego Księżyc zawsze pokazuje tę samą stronę Ziemi?
Księżyc pokazuje Ziemi zawsze tę samą stronę, ponieważ jest związany pływowo z Ziemią. Oznacza to, że jego czas obrotu wokół własnej osi jest równy czasowi obiegu wokół Ziemi (około 27,3 dnia). W wyniku działania sił grawitacyjnych Ziemi rotacja Księżyca została w przeszłości spowolniona aż do osiągnięcia tego stanu równowagi.
Now use the following context items to answer this one user query only:
{context}
Relevant passages: <extract relevant passages from the context here>
Main User Query: {query}
Answer:\n"""

# Wybieramy losowo zapytanie z listy
random_query = np.random.choice(questions)

response, chunks = utils.answer_question(
        prompt_template=prompt_template,
        query=random_query,
        max_tokens=512,
        temp=0.1
)

display(Markdown(f"**Pytanie:** {random_query}"))
display(Markdown(f"**Odpowiedź:**\n\n{response}"))
display(Markdown("---\n**Źródła:**"))
for i, chunk in enumerate(chunks):
    excerpt = chunk['chunk'][:200].strip() + "..."
    display(Markdown(
        f"**[{i+1}]** `{chunk['filename']}` — strona {chunk['page_number'] + 1}\n\n"
        f"> {excerpt}"
    ))

**Pytanie:** Jakie przyszłe technologie mogą poprawić wykrywanie egzoplanet?

**Odpowiedź:**

Specjalnie zaprojektowane instrumenty naziemne (m.in. VLT–SPHERE i Gemini Planet Imager, ref. 53) mają być uruchomione w późnym 2013 roku – na początku 2014 roku, aby wyprzedzić obserwacje te. Są wyposażone w „ekstremalne” adaptacyjne optyki (aby zminimalizować rozproszenie atmosferyczne intensywnego światła gwiazdy-sfitrióna) i koronografy (do tłumienia ).


---
**Źródła:**

**[1]** `Exoplanets.pdf` — strona 7

> nd bibliography through to the end of 2010. (I) 50. exoplanet.eu/catalog and select ‘detected by transits’ to identify the exoplanets currently detected by this method, and the bibliography for each....

**[2]** `Exoplanets.pdf` — strona 7

> , A. L´eger et al., Experimental Astronomy 23, 435–461 (2009). A summary of the scientiﬁc and design goals of the Darwin mission for exoplanet imaging from space, now in hibernation. (I) 56. Ref. 3, C...

**[3]** `Exoplanets.pdf` — strona 7

> the infrared spectral range). As a result only young, warm and self-luminous giant planets moving in a very wide (long-period) orbits have been detected to date. Specially-designed ground-based instr...

**[4]** `Exoplanets.pdf` — strona 8

> 8 59. “Very large optics for the study of extrasolar terrestrial plan- ets: Life Finder,” N.J. Woolf, Study Report, NASA Institute for Advanced Concepts (NASA, Atlanta, 2001). (I) 60. “Resolved imagin...

**[5]** `Exoplanets.pdf` — strona 6

> , ARA&A 50, 411–453 (2012). Review of the concepts of microlensing planet searches and their practical application. (I) 34. Ref. 3, Chapter 5 provides details of the principles, discov- eries, and bib...